# 08 · Ionisation of a strongly driven transmon (Sec. VII, Figs. 11–12)

Full Eq. (70) with $L=\sqrt\kappa\,a$, $\kappa/2\pi=20$ MHz, drive $\epsilon_d/2\pi=280$ MHz at $\omega_d$ from Eq. (74), starting from the dressed states $|\overline{0,0}\rangle$, $|\overline{1,0}\rangle$, $|\overline{2,0}\rangle$. The paper uses the transmon **eigenbasis** on $R_t=5$ bits and the cavity Fock basis on $R_c=10$ bits, tolerance $10^{-8}$, and reports $\chi_q<60$ and $\chi_\mu\approx120$.

**Frame.** We integrate in the frame $U=e^{i\omega_d t(K+a^\dagger a)}$ ($K$ = transmon level index). This is an exact change of variables — no rotating-wave approximation, the counter-rotating terms survive as time-dependent phases — but it lowers the largest frequency the integrator must resolve by about an order of magnitude, and it is the frame in which the Wigner functions of Fig. 11 are stationary blobs. In the QTT solver the same trick means splitting $\hat n_t$ into its diagonals $k-l=d$, each with coefficient $e^{id\omega_d t}$.

**Sizes.** With the QTT solver (`PAPER_SIZE=True`): $2^5\times2^{10}$ and $\kappa t$ up to 3. Without it: 16 transmon levels × 48 photons and $\kappa t\approx 0.8$ ($\langle N_c\rangle\lesssim 20$). That is enough to see the $|\overline{2,0}\rangle$ trajectory jump at the crossing its branch meets near $N_c\approx 10$–15 (notebook 07), but not the $|\overline{1,0}\rangle$ ionisation near $N_c\approx 50$.

**Cutoff diagnostic.** If you push the dense run further, $\chi_\mu$ suddenly explodes and the purity drops once the cavity distribution reaches the Fock cutoff (we saw $\chi_\mu$: 9 → 174 as $\langle N_c\rangle$ went from 21 to 35 with 48 photons). That is a truncation artefact, not ionisation. The run cell below warns when $\langle N_c\rangle+4\sqrt{\langle N_c\rangle}$ exceeds the cutoff; the same check belongs in any QTT run.

In [ ]:
import os, sys
from tutorial_utils import *
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
print(backend_banner())
PAPER_SIZE = HAVE_QTTLIND and os.environ.get("QTTLIND_PAPER_SIZE") == "1"   # published sizes (hours); default: quick mode

In [ ]:
import scipy.sparse.linalg as sla
if PAPER_SIZE:
    n_t, n_c, t_final = 32, 1024, 3.0 / (2 * np.pi * TRANSMON_PARAMS["kappa"])
else:
    n_t, n_c, t_final = 16, 48, 6.5            # ns  (kappa t ~ 0.8, <N_c> stays below ~20)
model = transmon_cavity_model(n_t, n_c if not PAPER_SIZE else 64)   # dressed states: cavity nearly empty
lab, vecs = dressed_states(model, n_branches=3, n_c_max=2)
wd = 0.5 * sum(lab[(i, 1)][0] - lab[(i, 0)][0] for i in range(2))
print(f"omega_d/2pi = {wd / (2 * np.pi):.4f} GHz, t_final = {t_final:.2f} ns (kappa t = {2*np.pi*TRANSMON_PARAMS['kappa']*t_final:.2f})")

## Run

In [ ]:
runs = {}
if HAVE_QTTLIND and PAPER_SIZE:
    sysq = ql.System()
    T = sysq.add_mode("transmon", bits=5,
                      basis=ql.CustomBasis(energies=2 * np.pi * model["Et"][:n_t], ops={"n": model["nt"][:n_t, :n_t]}))
    C = sysq.add_mode("cavity", bits=10)
    tw = 2 * np.pi; p = TRANSMON_PARAMS
    # exact drive frame: n_t split by diagonal offset d = k - l, each term with phase exp(i d w_d t)
    H = T.H0() - wd * T.level() + (tw * p["fr"] - wd) * C.n()
    for d in T.op_offsets("n"):
        H += ql.coeff(lambda t, d=d: np.exp(1j * (d - 1) * wd * t)) * (-1j * tw * p["g"]) * T.op("n", offset=d) * C
        H += ql.coeff(lambda t, d=d: np.exp(1j * (d + 1) * wd * t)) * (1j * tw * p["g"]) * T.op("n", offset=d) * C.dag()
    H += ql.coeff(lambda t: model["eps_d"] * np.sin(wd * t) * np.exp(-1j * wd * t)) * (-1j * C)
    H += ql.coeff(lambda t: model["eps_d"] * np.sin(wd * t) * np.exp(1j * wd * t)) * (1j * C.dag())
    sol = ql.LindbladSolver(sysq, H, [np.sqrt(tw * p["kappa"]) * C], scheme="kraus2", dt=2e-4, tol=1e-8)
    for it in range(3):
        psi0 = sysq.state_from_dense(vecs[(it, 0)].reshape(n_t, -1), pad={"cavity": 1024})
        runs[it] = sol.run(psi0, t_final=t_final, every=200, e_ops={"Nt": T.level(), "Nc": C.n()},
                           track=["purity", "chi_mu", "chi_q", "reduced_dm:cavity"])
else:
    heff_t, Ls = transmon_cavity_drive_frame(model, wd)
    E = abs(sla.eigs(heff_t(0.0), k=1, which="LM", return_eigenvectors=False)[0])
    h = 0.15 / E                                  # keep h*E small: Taylor propagators (guide, Phase 7)
    nsteps = int(round(t_final / h)); every = max(1, nsteps // 120)
    print(f"||H_eff|| ~ {E:.0f} rad/ns -> h = {h:.2e} ns, {nsteps} steps per initial state")
    for it in range(3):
        st = dr.Purified.from_ket(vecs[(it, 0)])
        out = {"t": [], "Nt": [], "Nc": [], "purity": [], "chi_mu": [], "rho_c": []}
        with timer(f"initial state |{it},0>"):
            for k in range(nsteps + 1):
                if k % every == 0:
                    out["t"].append(k * h); out["Nt"].append(st.expect(model["Nt"]).real)
                    out["Nc"].append(st.expect(model["Nc"]).real); out["purity"].append(st.purity())
                    out["chi_mu"].append(st.chi_mu)
                    out["rho_c"].append(reduced_dm(st.Psi, model["dims"], keep=1))
                    nc_ = out["Nc"][-1]
                    if nc_ + 4 * np.sqrt(nc_) > n_c and not out.get("warned"):
                        print(f"  WARNING t={k*h:.2f} ns: <N_c>={nc_:.1f} approaches the cutoff {n_c}; results beyond are truncation-dominated")
                        out["warned"] = True
                if k < nsteps:
                    dr.kraus_step(st, heff_t(k * h + h / 2), Ls, h, order=2, rtol=1e-8)
        out.pop("warned", None)
        runs[it] = {k_: np.array(v) for k_, v in out.items()}
        print(f"  final <N_c> = {out['Nc'][-1]:.1f}, <N_t> = {out['Nt'][-1]:.2f}, purity = {out['purity'][-1]:.4f}, chi_mu = {out['chi_mu'][-1]}")

### Quick-mode cross-check: the QTT solver on the driven transmon

The runs above use the dense oracle in quick mode. The cell below builds the **same drive-frame
Hamiltonian with the operator algebra** — a custom transmon basis, its charge operator split by
diagonal offset with time-dependent phases — and runs `ql.LindbladSolver` for 300 steps on
$8\times16$ levels, against the oracle with the same Kraus scheme.

In [ ]:
checks = globals().get("checks", {})
if HAVE_QTTLIND and not PAPER_SIZE:
    nt_q, nc_q, nq = 8, 16, 300
    mq = transmon_cavity_model(nt_q, nc_q)
    _, vq = dressed_states(mq, n_branches=2, n_c_max=1)
    heff_q, Ls_q = transmon_cavity_drive_frame(mq, wd)
    Eq = abs(sla.eigs(heff_q(0.0), k=1, which="LM", return_eigenvectors=False)[0])
    hq = 0.15 / Eq
    tw = 2 * np.pi; p = TRANSMON_PARAMS
    sysq = ql.System()
    T = sysq.add_mode("transmon", bits=3, basis=ql.CustomBasis(energies=tw * mq["Et"], ops={"n": mq["nt"]}))
    C = sysq.add_mode("cavity", bits=4)
    H = T.H0() - wd * T.level() + (tw * p["fr"] - wd) * C.n()
    for d in T.op_offsets("n"):
        H += ql.coeff(lambda t, d=d: np.exp(1j * (d - 1) * wd * t)) * (-1j * tw * p["g"]) * T.op("n", offset=d) * C
        H += ql.coeff(lambda t, d=d: np.exp(1j * (d + 1) * wd * t)) * (1j * tw * p["g"]) * T.op("n", offset=d) * C.dag()
    H += ql.coeff(lambda t: mq["eps_d"] * np.sin(wd * t) * np.exp(-1j * wd * t)) * (-1j * C)
    H += ql.coeff(lambda t: mq["eps_d"] * np.sin(wd * t) * np.exp(1j * wd * t)) * (1j * C.dag())
    sol = ql.LindbladSolver(sysq, H, [np.sqrt(tw * p["kappa"]) * C], scheme="kraus2", dt=hq, tol=1e-12)
    psi0 = sysq.state_from_dense(vq[(1, 0)].reshape(nt_q, nc_q))
    with timer("QTT solver, 300 steps"):
        rq = sol.run(psi0, t_final=nq * hq, every=30, e_ops={"Nt": T.level(), "Nc": C.n()}, track=["purity"])
    st = dr.Purified.from_ket(vq[(1, 0)])
    ref = {"Nt": [], "Nc": [], "purity": []}
    with timer("dense oracle, 300 steps"):
        for k in range(nq + 1):
            if k % 30 == 0:
                ref["Nt"].append(st.expect(mq["Nt"]).real); ref["Nc"].append(st.expect(mq["Nc"]).real)
                ref["purity"].append(st.purity())
            if k < nq:
                dr.kraus_step(st, heff_q(k * hq + hq / 2), Ls_q, hq, order=2, rtol=1e-12)
    err = max(np.abs(np.real(rq.expect["Nt"]) - ref["Nt"]).max(), np.abs(np.real(rq.expect["Nc"]) - ref["Nc"]).max(),
              np.abs(np.asarray(rq.track["purity"]) - ref["purity"]).max())
    print(f"h = {hq:.2e} ns; max deviation QTT vs oracle (N_t, N_c, purity) = {err:.1e}")
    checks["transmon: QTT vs oracle"] = (err, 1e-6)

## Fig. 11 — cavity Wigner functions (log scale)

Columns: equally spaced times; rows: initial transmon state. For the ground state the cavity stays close to a coherent state; for the excited states it is "bananified" as highly excited transmon levels get populated. On the reduced dense run, a faint ring at the $10^{-4}$ level appears in the last column; we have not checked whether it is physical (counter-rotating terms are kept) or a cutoff effect — compare with a larger `n_c` before interpreting it.

In [ ]:
cols = 4
re = np.linspace(-13, 2, 90) if PAPER_SIZE else np.linspace(-7, 3, 70)
im = np.linspace(0, 15, 90) if PAPER_SIZE else np.linspace(-1, 8, 70)
fig, axs = plt.subplots(3, cols, figsize=(3 * cols, 8.2), sharex=True, sharey=True)
cmaps = ["Blues", "Oranges", "Reds"]
for it in range(3):
    r = runs[it]
    rhos = r["rho_c"] if isinstance(r, dict) else r.track["reduced_dm:cavity"]
    ts = r["t"] if isinstance(r, dict) else r.times
    idx = np.linspace(0, len(ts) - 1, cols).astype(int)
    for j, k in enumerate(idx):
        W = wigner(rhos[k], re, im)
        axs[it, j].pcolormesh(re, im, np.log10(np.clip(np.abs(W), 1e-6, None)), cmap=cmaps[it],
                              vmin=-6, vmax=np.log10(2 / np.pi), shading="auto")
        axs[it, j].set_title(rf"$|\overline{{{it},0}}\rangle$, t={ts[k]:.1f} ns", fontsize=9)
for ax in axs[-1]: ax.set_xlabel(r"Re $\alpha$")
for ax in axs[:, 0]: ax.set_ylabel(r"Im $\alpha$")
plt.tight_layout(); savefig(fig, "fig11_wigner.png"); plt.show()

## Fig. 12 — trajectories, purity and bond dimensions

In [ ]:
def get(r, key):
    return np.asarray(r[key] if isinstance(r, dict) else (r.expect.get(key, None) if key in ("Nt", "Nc") else r.track[key])).real

fig, axs = plt.subplots(2, 2, figsize=(11, 7.5))
kap = 2 * np.pi * TRANSMON_PARAMS["kappa"]
try:
    br = np.load(os.path.join(FIGDIR, "branches_Nt_Nc.npy"))          # from notebook 07
    for it in range(br.shape[0]):
        axs[0, 0].plot(br[it, :, 1], br[it, :, 0], color="0.8", lw=0.8)
except FileNotFoundError:
    print("run notebook 07 first to draw the branches in panel a")
for it, col in zip(range(3), ["C0", "C1", "C3"]):
    r = runs[it]
    ts = r["t"] if isinstance(r, dict) else r.times
    axs[0, 0].plot(get(r, "Nc"), get(r, "Nt"), color=col, label=rf"$|\overline{{{it},0}}\rangle$")
    axs[1, 0].plot(kap * ts, get(r, "purity"), color=col)
    axs[1, 1].plot(kap * ts, get(r, "chi_mu"), color=col)
    if not isinstance(r, dict):
        axs[0, 1].plot(kap * ts, r.track["chi_q"], color=col)
axs[0, 0].set(xlabel=r"$N_c$", ylabel=r"$N_t$", title="a", xlim=(0, max(get(runs[i], "Nc").max() for i in range(3)) * 1.1),
              ylim=(0, max(get(runs[i], "Nt").max() for i in range(3)) * 1.2)); axs[0, 0].legend()
axs[1, 0].set(xlabel=r"$\kappa t$", ylabel=r"Tr $\rho^2$", title="c")
axs[1, 1].set(xlabel=r"$\kappa t$", ylabel=r"$\chi_\mu$", title="d")
if isinstance(runs[0], dict):
    axs[0, 1].text(0.5, 0.5, r"$\chi_q(t)$ needs the QTT solver" "\n" "(PAPER_SIZE=True)", ha="center", va="center",
                   transform=axs[0, 1].transAxes)
axs[0, 1].set(xlabel=r"$\kappa t$", ylabel=r"$\chi_q$", title="b")
plt.tight_layout(); savefig(fig, "fig12_ionization.png"); plt.show()

What to compare with the paper:
* **a** — the blue trajectory follows its branch with a slow rise of $N_t$; the orange one jumps when $N_c\approx 50$ (needs the paper-size run); the red one is the paper's new result.
* **c, d** — the jump coincides with a drop of purity and a rise of $\chi_\mu$, which stays around 120 at paper size, far below $2^{15}$.
* **b** — $\chi_q<60$ and plateauing.